# Initialization

In [0]:
from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

In [0]:
"""
Rule-based cleaning of document body text (silver layer).
 
Scope: web text extraction (trafilatura) already strips most boilerplate.
This module only removes the leftovers it misses.
 
Granularity: line by line. Each line is first tagged with the rule that
removes it (removed_by); after filtering, the kept lines are reassembled
into full text by line number. The tagged lines double as the removal log.

"""

CLEANING_VERSION = "v2"   # Bump on any change to rules or parameters
MAX_CHARS = 12_000        # Max characters sent to the AI model

# ---------------------------------------------------------------------------
# Rules (matched against a single line with leading/trailing whitespace
# stripped; patterns never span lines)
#   - rlike matches if the pattern appears anywhere in the line;
#     use ^...$ to require a full-line match
#   - If a line matches several rules, the one listed first wins
#   - After adding a rule: add a test case under tests/ and bump
#     CLEANING_VERSION
# ---------------------------------------------------------------------------



### Tail rules and line rules

In [0]:
#Tail rules: the first matching line in a document and every line after it
# are removed

TAIL_RULES = [
    ("about_company", r"^About .{1,60}$"),     # "About Hyundai Mobis"
    ("media_contact", r"^Media Contacts?:?"),  # "Media Contact" / "Media Contacts:"
    ("source_line",   r"^SOURCE .+$"),         # "SOURCE Agility Robotics"
]
 
# Line rules: only the matching line itself is removed
LINE_RULES = [
    ("article_sidebar", r"^Article Sidebar$"),
    ("email_protected", r"\[email\s*protected\]"),
    ("recruiting",      r"please consider applying"),
]

### Define cleaning functions

In [0]:
# first_matching_rule

def first_matching_rule(text: Column, rules: list[tuple[str, str]]) -> Column:
    """
    Return the name of the first matching rule, or null if none match.
    
    """
    if rules:
         return F.coalesce(*[F.when(text.rlike(pattern), F.lit(name)) for name, pattern in rules])
    else:
         return F.lit(None).cast("string")    
     
    

# split_into_lines

def split_into_lines(df_docs: DataFrame) -> DataFrame:
    return df_docs.select(
        "doc_id",
        F.posexplode(F.split("raw_text", r"\r?\n")).alias("line_no", "line")
    )
    
# flag_lines

def flag_lines(df_lines: DataFrame) -> DataFrame:
    trimmed = F.trim("line")
    df = (
        df_lines
        .withColumn("tail_rule", first_matching_rule(trimmed, TAIL_RULES))
        .withColumn("line_rule", first_matching_rule(trimmed, LINE_RULES))
    )

    tail_start = (
        df
        .filter(F.col("tail_rule").isNotNull())
        .groupby("doc_id")
        .agg(F.min(F.struct("line_no", "tail_rule")).alias("first_tail"))
        .select(
            "doc_id",
            F.col("first_tail.line_no").alias("tail_start_line"),
            F.col("first_tail.tail_rule").alias("tail_start_rule"),
        )
    )
    
    return (
        df
        .join(tail_start, "doc_id", "left")
        .withColumn("removed_reason",
                F.when(F.col("line_no")>=F.col("tail_start_line"),
                       F.concat(F.lit("tail:"), F.col("tail_start_rule")))
                .when(F.col("line_rule").isNotNull(),
                      F.concat(F.lit("line:"), F.col("line_rule"))),
                )
        .select("doc_id", "line_no", "line", "removed_reason")
    )

# assemble_documents

def assemble_documents(df_flagged: DataFrame) -> DataFrame:
    kept_line = F.when(
                  F.col("removed_reason").isNull(), F.struct("line_no", "line"))
    return df_flagged.groupby("doc_id").agg(
        F.array_join(
            F.transform(F.sort_array(F.collect_list(kept_line)), 
                        lambda x: x["line"]),
            "\n",
        ).alias("clean_text"),
        F.count("*").alias("raw_line_count"),
        F.sum(F.col("removed_reason").isNotNull().cast("int")).alias("removed_line_count"),
    )

# tidy_whitespace

def tidy_whitespace(text:Column) -> Column:
    return F.trim(F.regexp_replace(text, r"\n{3,}", "\n\n"))


def build_documents_clean(
    df_docs: DataFrame, df_flagged: DataFrame, max_chars: int = MAX_CHARS
) -> DataFrame:
    raw_lengths = df_docs.select(
        "doc_id", "language", F.length("raw_text").alias("raw_char_count"))

    return (
        assemble_documents(df_flagged)
        .withColumn("clean_text", tidy_whitespace(F.col("clean_text")))
        .join(raw_lengths, "doc_id")
        .withColumn("cleaned_char_count", F.length("clean_text"))
        .withColumn(
            "removed_ratio",
            F.when(F.col("raw_char_count")>0,
                   F.round(1 - F.col("cleaned_char_count")/F.col("raw_char_count"), 2)),
        )
        .withColumn("is_truncated", F.col("cleaned_char_count")>max_chars)
        .withColumn("clean_text", F.substring("clean_text", 1, max_chars))
        .withColumn("cleaning_version", F.lit(CLEANING_VERSION))
        .withColumn("cleaned_time", F.current_timestamp())
        .select(
            "doc_id", "clean_text",
            "raw_char_count", "cleaned_char_count", "removed_ratio",
            "raw_line_count", "removed_line_count", "is_truncated", "language", "cleaning_version", "cleaned_time",

        )
    )

